# Analisis Pengaruh Kemiskinan dan Sanitasi terhadap Prevalensi Stunting di Indonesia
## Menggunakan Regresi Linear dan K-Means Clustering
### Studi 34 Provinsi Tahun 2021–2024

---

**Metodologi:** CRISP-DM (Cross-Industry Standard Process for Data Mining)

**Deskripsi:**
Notebook ini menganalisis hubungan antara **kemiskinan**, **akses sanitasi layak**, dan **prevalensi stunting** pada balita di 34 provinsi Indonesia selama periode 2021–2024. Analisis menggunakan pendekatan CRISP-DM yang terdiri dari 6 fase:
1. *Business Understanding*
2. *Data Understanding*
3. *Data Preparation*
4. *Modeling*
5. *Evaluation*
6. *Deployment*

**Dataset yang digunakan:**
1. Data Prevalensi Stunting Balita per Provinsi (Kemenkes RI)
2. Data Jumlah & Persentase Penduduk Miskin per Provinsi (BPS)
3. Data Persentase Rumah Tangga dengan Akses Sanitasi Layak per Provinsi (BPS)


## FASE 1: Business Understanding

### 1.1 Latar Belakang
Stunting merupakan masalah gizi kronis yang disebabkan oleh kurangnya asupan gizi dalam waktu lama, umumnya karena asupan makanan yang tidak sesuai kebutuhan gizi. Stunting menjadi salah satu fokus utama pembangunan Indonesia, dengan target penurunan prevalensi stunting menjadi 14% pada tahun 2024 (RPJMN 2020–2024).

Kemiskinan dan sanitasi merupakan dua faktor determinan yang diduga berpengaruh kuat terhadap stunting:
- **Kemiskinan** membatasi akses terhadap pangan bergizi, layanan kesehatan, dan lingkungan hidup yang layak.
- **Sanitasi** yang buruk meningkatkan risiko penyakit infeksi berulang yang mengganggu penyerapan nutrisi pada anak.

### 1.2 Pertanyaan Penelitian
1. Apakah terdapat korelasi signifikan antara kemiskinan, sanitasi, dan prevalensi stunting di Indonesia?
2. Seberapa akurat kemiskinan dan sanitasi dapat memprediksi prevalensi stunting?
3. Bagaimana pengelompokan provinsi berdasarkan profil kemiskinan, sanitasi, dan stunting?
4. Bagaimana tren kemiskinan, sanitasi, dan stunting selama 2021–2024?

### 1.3 Pertanyaan Analitik

| No | Pertanyaan Analitik | Metode |
|----|---------------------|--------|
| Q1 | Apakah kemiskinan dan sanitasi berkorelasi dengan stunting? | Korelasi Pearson |
| Q2 | Seberapa akurat kemiskinan dan sanitasi memprediksi stunting? | Regresi Linear Berganda |
| Q3 | Provinsi mana yang memiliki profil serupa? | K-Means Clustering (3D) |
| Q4 | Bagaimana tren 2021–2024? | Analisis Temporal / Line Plot |

### 1.4 Kriteria Keberhasilan

| No | Kriteria | Target |
|----|----------|--------|
| 1 | Korelasi Pearson signifikan (p < 0.05) antara kemiskinan/sanitasi & stunting | Ya |
| 2 | R² regresi linear ≥ 0.30 | ≥ 0.30 |
| 3 | Silhouette Score clustering ≥ 0.40 | ≥ 0.40 |
| 4 | Identifikasi tren temporal yang konsisten | Ya |


---
## FASE 2: Data Understanding

### 2.1 Import Library


In [ ]:
# Helper untuk pencarian path file otomatis di Colab maupun Lokal
def get_path(filename, folder="dataset/raw"):
    # Cek jika folder ada di direktori parent (jika berjalan dari dalam folder notebooks)
    if os.path.exists(os.path.join("..", folder)):
        return os.path.join("..", folder, filename)
    # Cek jika folder ada di direktori kerja saat ini (jika berjalan dari root)
    if os.path.exists(folder):
        return os.path.join(folder, filename)
    # Fallback ke filename langsung (misal di Google Colab tanpa subfolder)
    return filename


### 2.2 Memuat Dataset

#### 2.2.1 Dataset Stunting


In [ ]:
# Memuat data stunting
df_stunting = pd.read_csv(
    get_path('vertikalkementerian-2-od_20953_prevalensi_balita_stunting_brdsrkn_prov_di_indones_v1_data.csv', 'dataset/raw')
)

print("="*60)
print("DATA STUNTING - Informasi Awal")
print("="*60)
print(f"\nShape: {df_stunting.shape}")
print(f"\nKolom: {list(df_stunting.columns)}")
print(f"\nTipe data:\n{df_stunting.dtypes}")
print(f"\nMissing values:\n{df_stunting.isnull().sum()}")
print(f"\nTahun unik: {sorted(df_stunting['tahun'].unique())}")
print(f"\nJumlah provinsi per tahun:")
print(df_stunting.groupby('tahun')['nama_provinsi'].nunique())
print(f"\n5 baris pertama:")
df_stunting.head()


#### 2.2.2 Dataset Kemiskinan

In [ ]:
# Memuat data kemiskinan (4 file, satu per tahun)
dfs_kemiskinan = []
for tahun in range(2021, 2025):
    fp = f'Jumlah_dan_Persentase_Penduduk_Miskin_Menurut_Provinsi__{tahun}.csv'
    tmp = pd.read_csv(get_path(fp, 'dataset/raw'))
    tmp['tahun'] = tahun
    dfs_kemiskinan.append(tmp)

df_kemiskinan_raw = pd.concat(dfs_kemiskinan, ignore_index=True)

print("="*60)
print("DATA KEMISKINAN - Informasi Awal")
print("="*60)
print(f"\nShape: {df_kemiskinan_raw.shape}")
print(f"\nKolom: {list(df_kemiskinan_raw.columns)}")
print(f"\nTipe data:\n{df_kemiskinan_raw.dtypes}")
print(f"\nMissing values:\n{df_kemiskinan_raw.isnull().sum()}")
print(f"\nNilai unik Provinsi: {df_kemiskinan_raw['Provinsi'].nunique()}")
print(f"\n5 baris pertama:")
df_kemiskinan_raw.head()


#### 2.2.3 Dataset Sanitasi

In [ ]:
# Memuat data sanitasi (4 file, satu per tahun)
# Format: 3 baris header, lalu data dimulai dari baris ke-4
dfs_sanitasi = []
for tahun in range(2021, 2025):
    fp = f'Persentase Rumah Tangga menurut Provinsi dan Memiliki Akses terhadap Sanitasi Layak, {tahun}.csv'
    tmp = pd.read_csv(get_path(fp, 'dataset/raw'), header=None, skiprows=3)
    tmp.columns = ['provinsi', 'persen_sanitasi']
    tmp['tahun'] = tahun
    dfs_sanitasi.append(tmp)

df_sanitasi_raw = pd.concat(dfs_sanitasi, ignore_index=True)

print("="*60)
print("DATA SANITASI - Informasi Awal")
print("="*60)
print(f"\nShape: {df_sanitasi_raw.shape}")
print(f"\nKolom: {list(df_sanitasi_raw.columns)}")
print(f"\nTipe data:\n{df_sanitasi_raw.dtypes}")
print(f"\nMissing values:\n{df_sanitasi_raw.isnull().sum()}")
print(f"\nNilai unik provinsi: {df_sanitasi_raw['provinsi'].nunique()}")
print(f"\nContoh provinsi:")
print(df_sanitasi_raw['provinsi'].unique())
print(f"\n5 baris pertama:")
df_sanitasi_raw.head()


### 2.3 Identifikasi Masalah Kualitas Data

Berdasarkan eksplorasi awal, ditemukan beberapa masalah kualitas data:

| No | Dataset | Masalah | Keterangan |
|----|---------|---------|------------|
| 1 | Stunting | Provinsi pemekaran baru | PAPUA BARAT DAYA, PAPUA SELATAN, PAPUA TENGAH, PAPUA PEGUNUNGAN (tidak termasuk 34 provinsi standar) |
| 2 | Kemiskinan | Nilai `...` (junk) | Beberapa kolom September berisi `...` |
| 3 | Kemiskinan | Baris agregat `Indonesia` | Harus dihapus |
| 4 | Kemiskinan | Baris provinsi pemekaran | PAPUA BARAT DAYA, PAPUA SELATAN, dst. |
| 5 | Sanitasi | Format header tidak standar | 3 baris pertama adalah header non-standar |
| 6 | Sanitasi | Singkatan nama provinsi | `KEP. BANGKA BELITUNG` dan `KEP. RIAU` |
| 7 | Sanitasi | Nilai `-` (missing) | Beberapa provinsi baru bernilai `-` |
| 8 | Sanitasi | Baris INDONESIA & provinsi pemekaran | Harus dihapus |
| 9 | Semua | Inkonsistensi nama provinsi | Perlu normalisasi huruf besar |


### 2.4 Distribusi Awal (Data Mentah)

In [ ]:
# Distribusi awal data stunting
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Stunting
axes[0].hist(df_stunting['prevalensi_balita_stunting'], bins=20, color='#e74c3c',
             edgecolor='white', alpha=0.8)
axes[0].set_title('Distribusi Prevalensi Stunting (Raw)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Prevalensi Stunting (%)')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(df_stunting['prevalensi_balita_stunting'].mean(), color='black',
                linestyle='--', label=f"Mean: {df_stunting['prevalensi_balita_stunting'].mean():.1f}")
axes[0].legend()

# Kemiskinan - kita ambil kolom Persentase Penduduk Miskin - Maret
kem_vals = pd.to_numeric(df_kemiskinan_raw['Persentase Penduduk Miskin - Maret'],
                         errors='coerce').dropna()
axes[1].hist(kem_vals, bins=20, color='#3498db', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribusi Persentase Kemiskinan (Raw)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Persentase Penduduk Miskin (%)')
axes[1].set_ylabel('Frekuensi')
axes[1].axvline(kem_vals.mean(), color='black', linestyle='--',
                label=f"Mean: {kem_vals.mean():.1f}")
axes[1].legend()

# Sanitasi
san_vals = pd.to_numeric(df_sanitasi_raw['persen_sanitasi'], errors='coerce').dropna()
axes[2].hist(san_vals, bins=20, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[2].set_title('Distribusi Persentase Sanitasi Layak (Raw)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Persentase Sanitasi Layak (%)')
axes[2].set_ylabel('Frekuensi')
axes[2].axvline(san_vals.mean(), color='black', linestyle='--',
                label=f"Mean: {san_vals.mean():.1f}")
axes[2].legend()

plt.tight_layout()
plt.show()


---
## FASE 3: Data Preparation

### 3.1 Daftar 34 Provinsi Standar


In [ ]:
# 34 provinsi standar Indonesia (sebelum pemekaran Papua)
PROVINSI_34 = [
    'ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'JAMBI',
    'SUMATERA SELATAN', 'BENGKULU', 'LAMPUNG',
    'KEPULAUAN BANGKA BELITUNG', 'KEPULAUAN RIAU',
    'DKI JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'DI YOGYAKARTA',
    'JAWA TIMUR', 'BANTEN', 'BALI',
    'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR',
    'KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN',
    'KALIMANTAN TIMUR', 'KALIMANTAN UTARA',
    'SULAWESI UTARA', 'SULAWESI TENGAH', 'SULAWESI SELATAN',
    'SULAWESI TENGGARA', 'GORONTALO', 'SULAWESI BARAT',
    'MALUKU', 'MALUKU UTARA',
    'PAPUA BARAT', 'PAPUA'
]
print(f"Jumlah provinsi standar: {len(PROVINSI_34)}")


### 3.2 Cleaning Data Stunting

In [ ]:
# Cleaning data stunting
df_stunting_clean = df_stunting.copy()

# Uppercase nama provinsi
df_stunting_clean['nama_provinsi'] = df_stunting_clean['nama_provinsi'].str.strip().str.upper()

# Filter hanya 34 provinsi standar
df_stunting_clean = df_stunting_clean[
    df_stunting_clean['nama_provinsi'].isin(PROVINSI_34)
].copy()

# Rename kolom agar konsisten
df_stunting_clean = df_stunting_clean.rename(columns={
    'nama_provinsi': 'provinsi',
    'prevalensi_balita_stunting': 'prevalensi_stunting'
})

# Ambil kolom yang dibutuhkan saja
df_stunting_clean = df_stunting_clean[['provinsi', 'prevalensi_stunting', 'tahun']].copy()

print("="*60)
print("DATA STUNTING - Setelah Cleaning")
print("="*60)
print(f"Shape: {df_stunting_clean.shape}")
print(f"Provinsi unik: {df_stunting_clean['provinsi'].nunique()}")
print(f"Tahun: {sorted(df_stunting_clean['tahun'].unique())}")
print(f"Missing values:\n{df_stunting_clean.isnull().sum()}")
print(f"\nStatistik deskriptif:")
df_stunting_clean.describe()


### 3.3 Cleaning Data Kemiskinan

In [ ]:
# Cleaning data kemiskinan
df_kem_clean = df_kemiskinan_raw.copy()

# Uppercase nama provinsi
df_kem_clean['Provinsi'] = df_kem_clean['Provinsi'].str.strip().str.upper()

# Hapus baris INDONESIA (agregat) dan provinsi pemekaran baru
prov_exclude = ['INDONESIA', 'PAPUA BARAT DAYA', 'PAPUA SELATAN',
                'PAPUA TENGAH', 'PAPUA PEGUNUNGAN']
df_kem_clean = df_kem_clean[~df_kem_clean['Provinsi'].isin(prov_exclude)].copy()

# Filter hanya 34 provinsi standar
df_kem_clean = df_kem_clean[df_kem_clean['Provinsi'].isin(PROVINSI_34)].copy()

# Ganti '...' dengan NaN dan konversi ke numerik (kolom Maret saja)
cols_maret = [
    'Garis Kemiskinan - Maret (Rp)',
    'Jumlah Penduduk Miskin - Maret (ribu) (Ribu)',
    'Persentase Penduduk Miskin - Maret'
]

for col in cols_maret:
    df_kem_clean[col] = df_kem_clean[col].replace('...', np.nan)
    df_kem_clean[col] = pd.to_numeric(df_kem_clean[col], errors='coerce')

# Ambil kolom yang dibutuhkan
df_kem_clean = df_kem_clean.rename(columns={
    'Provinsi': 'provinsi',
    'Garis Kemiskinan - Maret (Rp)': 'garis_kemiskinan',
    'Jumlah Penduduk Miskin - Maret (ribu) (Ribu)': 'jumlah_miskin_ribu',
    'Persentase Penduduk Miskin - Maret': 'persen_miskin'
})

df_kem_clean = df_kem_clean[['provinsi', 'garis_kemiskinan', 'jumlah_miskin_ribu',
                              'persen_miskin', 'tahun']].copy()

print("="*60)
print("DATA KEMISKINAN - Setelah Cleaning")
print("="*60)
print(f"Shape: {df_kem_clean.shape}")
print(f"Provinsi unik: {df_kem_clean['provinsi'].nunique()}")
print(f"Tahun: {sorted(df_kem_clean['tahun'].unique())}")
print(f"Missing values:\n{df_kem_clean.isnull().sum()}")
print(f"\nStatistik deskriptif:")
df_kem_clean.describe()


### 3.4 Cleaning Data Sanitasi

In [ ]:
# Cleaning data sanitasi
df_san_clean = df_sanitasi_raw.copy()

# Uppercase dan strip
df_san_clean['provinsi'] = df_san_clean['provinsi'].str.strip().str.upper()

# Normalisasi singkatan nama provinsi
name_map_sanitasi = {
    'KEP. BANGKA BELITUNG': 'KEPULAUAN BANGKA BELITUNG',
    'KEP. RIAU': 'KEPULAUAN RIAU'
}
df_san_clean['provinsi'] = df_san_clean['provinsi'].replace(name_map_sanitasi)

# Hapus INDONESIA dan provinsi pemekaran baru
prov_exclude_san = ['INDONESIA', 'PAPUA BARAT DAYA', 'PAPUA SELATAN',
                    'PAPUA TENGAH', 'PAPUA PEGUNUNGAN']
df_san_clean = df_san_clean[~df_san_clean['provinsi'].isin(prov_exclude_san)].copy()

# Filter hanya 34 provinsi standar
df_san_clean = df_san_clean[df_san_clean['provinsi'].isin(PROVINSI_34)].copy()

# Ganti '-' dengan NaN dan konversi ke numerik
df_san_clean['persen_sanitasi'] = df_san_clean['persen_sanitasi'].replace('-', np.nan)
df_san_clean['persen_sanitasi'] = pd.to_numeric(df_san_clean['persen_sanitasi'], errors='coerce')

print("="*60)
print("DATA SANITASI - Setelah Cleaning")
print("="*60)
print(f"Shape: {df_san_clean.shape}")
print(f"Provinsi unik: {df_san_clean['provinsi'].nunique()}")
print(f"Tahun: {sorted(df_san_clean['tahun'].unique())}")
print(f"Missing values:\n{df_san_clean.isnull().sum()}")
print(f"\nStatistik deskriptif:")
df_san_clean.describe()


### 3.5 Merge Dataset

In [ ]:
# Merge ketiga dataset berdasarkan provinsi dan tahun
df_merged = df_stunting_clean.merge(
    df_kem_clean, on=['provinsi', 'tahun'], how='inner'
).merge(
    df_san_clean, on=['provinsi', 'tahun'], how='inner'
)

print("="*60)
print("DATASET GABUNGAN (MERGED)")
print("="*60)
print(f"Shape: {df_merged.shape}")
print(f"Provinsi unik: {df_merged['provinsi'].nunique()}")
print(f"Tahun unik: {sorted(df_merged['tahun'].unique())}")
print(f"\nMissing values setelah merge:")
print(df_merged.isnull().sum())
print(f"\nJumlah data per tahun:")
print(df_merged.groupby('tahun').size())
print(f"\n5 baris pertama:")
df_merged.head()


In [ ]:
# Tampilkan statistik deskriptif dataset final
print("="*60)
print("STATISTIK DESKRIPTIF - DATASET FINAL")
print("="*60)
df_merged[['prevalensi_stunting', 'persen_miskin', 'persen_sanitasi',
           'garis_kemiskinan', 'jumlah_miskin_ribu']].describe().round(2)


In [ ]:
# Simpan dataset final
df_merged.to_csv(get_path('dataset_final_kemiskinan_stunting.csv', 'dataset/processed'), index=False)
print("Dataset final berhasil disimpan: dataset_final_kemiskinan_stunting.csv")
print(f"Total baris: {len(df_merged)}, Total kolom: {len(df_merged.columns)}")


---
## FASE 4: Modeling

### 4.1 Analisis Korelasi (Pearson)


In [ ]:
# Korelasi Pearson antara 3 variabel utama
cols_corr = ['prevalensi_stunting', 'persen_miskin', 'persen_sanitasi']
corr_matrix = df_merged[cols_corr].corr(method='pearson')

print("="*60)
print("MATRIKS KORELASI PEARSON")
print("="*60)
print(corr_matrix.round(4))

# Uji signifikansi korelasi
pairs = [
    ('persen_miskin', 'prevalensi_stunting'),
    ('persen_sanitasi', 'prevalensi_stunting'),
    ('persen_miskin', 'persen_sanitasi')
]

print("\n" + "="*60)
print("UJI SIGNIFIKANSI KORELASI")
print("="*60)
for x_col, y_col in pairs:
    r, p = stats.pearsonr(df_merged[x_col].dropna(), df_merged[y_col].dropna())
    sig = "SIGNIFIKAN" if p < 0.05 else "TIDAK SIGNIFIKAN"
    print(f"\n{x_col} vs {y_col}:")
    print(f"  Pearson r = {r:.4f}, p-value = {p:.6f} -> {sig}")


#### 4.1.1 Heatmap Korelasi

In [ ]:
# Heatmap korelasi
fig, ax = plt.subplots(figsize=(8, 6))

labels = ['Stunting (%)', 'Kemiskinan (%)', 'Sanitasi (%)']
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1,
            xticklabels=labels, yticklabels=labels,
            square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 14, 'fontweight': 'bold'})
ax.set_title('Heatmap Korelasi Pearson\nStunting, Kemiskinan, & Sanitasi',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


#### 4.1.2 Scatter Plot

In [ ]:
# Scatter plot: Kemiskinan vs Stunting dan Sanitasi vs Stunting
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Kemiskinan vs Stunting
sns.regplot(data=df_merged, x='persen_miskin', y='prevalensi_stunting',
            ax=axes[0], scatter_kws={'alpha': 0.5, 'color': '#e74c3c'},
            line_kws={'color': '#c0392b', 'linewidth': 2})
r1, p1 = stats.pearsonr(df_merged['persen_miskin'], df_merged['prevalensi_stunting'])
axes[0].set_title(f'Kemiskinan vs Stunting\nr={r1:.3f}, p={p1:.4f}',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Persentase Penduduk Miskin (%)', fontsize=11)
axes[0].set_ylabel('Prevalensi Stunting (%)', fontsize=11)

# Plot 2: Sanitasi vs Stunting
sns.regplot(data=df_merged, x='persen_sanitasi', y='prevalensi_stunting',
            ax=axes[1], scatter_kws={'alpha': 0.5, 'color': '#2ecc71'},
            line_kws={'color': '#27ae60', 'linewidth': 2})
r2, p2 = stats.pearsonr(df_merged['persen_sanitasi'], df_merged['prevalensi_stunting'])
axes[1].set_title(f'Sanitasi vs Stunting\nr={r2:.3f}, p={p2:.4f}',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Persentase Sanitasi Layak (%)', fontsize=11)
axes[1].set_ylabel('Prevalensi Stunting (%)', fontsize=11)

plt.tight_layout()
plt.show()


### 4.2 Regresi Linear

#### 4.2.1 Regresi Sederhana: Kemiskinan → Stunting


In [ ]:
# Regresi sederhana: Kemiskinan -> Stunting
X_kem = df_merged[['persen_miskin']].values
y = df_merged['prevalensi_stunting'].values

reg_kem = LinearRegression()
reg_kem.fit(X_kem, y)

y_pred_kem = reg_kem.predict(X_kem)
r2_kem = r2_score(y, y_pred_kem)
rmse_kem = np.sqrt(mean_squared_error(y, y_pred_kem))
mae_kem = mean_absolute_error(y, y_pred_kem)

print("="*60)
print("REGRESI SEDERHANA: Kemiskinan -> Stunting")
print("="*60)
print(f"Persamaan: Stunting = {reg_kem.coef_[0]:.4f} * Kemiskinan + {reg_kem.intercept_:.4f}")
print(f"R² Score : {r2_kem:.4f}")
print(f"RMSE     : {rmse_kem:.4f}")
print(f"MAE      : {mae_kem:.4f}")
print(f"\nInterpretasi:")
print(f"  Setiap kenaikan 1% kemiskinan, stunting naik {reg_kem.coef_[0]:.2f}%")


#### 4.2.2 Regresi Sederhana: Sanitasi → Stunting

In [ ]:
# Regresi sederhana: Sanitasi -> Stunting
X_san = df_merged[['persen_sanitasi']].values

reg_san = LinearRegression()
reg_san.fit(X_san, y)

y_pred_san = reg_san.predict(X_san)
r2_san = r2_score(y, y_pred_san)
rmse_san = np.sqrt(mean_squared_error(y, y_pred_san))
mae_san = mean_absolute_error(y, y_pred_san)

print("="*60)
print("REGRESI SEDERHANA: Sanitasi -> Stunting")
print("="*60)
print(f"Persamaan: Stunting = {reg_san.coef_[0]:.4f} * Sanitasi + {reg_san.intercept_:.4f}")
print(f"R² Score : {r2_san:.4f}")
print(f"RMSE     : {rmse_san:.4f}")
print(f"MAE      : {mae_san:.4f}")
print(f"\nInterpretasi:")
print(f"  Setiap kenaikan 1% sanitasi, stunting berubah {reg_san.coef_[0]:.2f}%")


#### 4.2.3 Regresi Linear Berganda: Kemiskinan + Sanitasi → Stunting

In [ ]:
# Regresi linear berganda: Kemiskinan + Sanitasi -> Stunting
X_multi = df_merged[['persen_miskin', 'persen_sanitasi']].values

reg_multi = LinearRegression()
reg_multi.fit(X_multi, y)

y_pred_multi = reg_multi.predict(X_multi)
r2_multi = r2_score(y, y_pred_multi)
rmse_multi = np.sqrt(mean_squared_error(y, y_pred_multi))
mae_multi = mean_absolute_error(y, y_pred_multi)

print("="*60)
print("REGRESI LINEAR BERGANDA: Kemiskinan + Sanitasi -> Stunting")
print("="*60)
print(f"Persamaan: Stunting = {reg_multi.coef_[0]:.4f} * Kemiskinan + "
      f"{reg_multi.coef_[1]:.4f} * Sanitasi + {reg_multi.intercept_:.4f}")
print(f"R² Score : {r2_multi:.4f}")
print(f"RMSE     : {rmse_multi:.4f}")
print(f"MAE      : {mae_multi:.4f}")
print(f"\nKoefisien:")
print(f"  Kemiskinan : {reg_multi.coef_[0]:.4f}")
print(f"  Sanitasi   : {reg_multi.coef_[1]:.4f}")
print(f"  Intercept  : {reg_multi.intercept_:.4f}")


#### 4.2.4 Perbandingan Model Regresi

In [ ]:
# Perbandingan R² ketiga model
comparison = pd.DataFrame({
    'Model': ['Kemiskinan -> Stunting',
              'Sanitasi -> Stunting',
              'Kemiskinan + Sanitasi -> Stunting'],
    'R²': [r2_kem, r2_san, r2_multi],
    'RMSE': [rmse_kem, rmse_san, rmse_multi],
    'MAE': [mae_kem, mae_san, mae_multi]
}).round(4)

print("="*60)
print("PERBANDINGAN MODEL REGRESI")
print("="*60)
print(comparison.to_string(index=False))

# Visualisasi perbandingan R²
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(comparison['Model'], comparison['R²'],
              color=['#e74c3c', '#2ecc71', '#3498db'], edgecolor='white', width=0.6)
ax.set_title('Perbandingan R² Score Antar Model Regresi', fontsize=14, fontweight='bold')
ax.set_ylabel('R² Score', fontsize=12)
ax.set_ylim(0, 1)
for bar, val in zip(bars, comparison['R²']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
plt.xticks(fontsize=10)
plt.tight_layout()
plt.show()

# Simpan model terbaik (regresi berganda)
joblib.dump(reg_multi, get_path('model_regresi_stunting.pkl', 'models'))
print("\nModel regresi berganda disimpan: model_regresi_stunting.pkl")


#### 4.2.5 Visualisasi Prediksi vs Aktual

In [ ]:
# Plot prediksi vs aktual untuk model terbaik
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y, y_pred_multi, alpha=0.5, color='#3498db', edgecolors='white', s=60)
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Stunting Aktual (%)', fontsize=12)
ax.set_ylabel('Stunting Prediksi (%)', fontsize=12)
ax.set_title(f'Prediksi vs Aktual - Regresi Berganda\nR² = {r2_multi:.4f}',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.show()


### 4.3 K-Means Clustering (3D)

#### 4.3.1 Persiapan Data untuk Clustering


In [ ]:
# Rata-rata per provinsi (4 tahun) untuk clustering
df_avg = df_merged.groupby('provinsi').agg({
    'persen_miskin': 'mean',
    'prevalensi_stunting': 'mean',
    'persen_sanitasi': 'mean'
}).reset_index().round(2)

print("="*60)
print("DATA RATA-RATA PER PROVINSI (untuk Clustering)")
print("="*60)
print(f"Shape: {df_avg.shape}")
print(f"\nStatistik deskriptif:")
print(df_avg.describe().round(2))
print(f"\n5 baris pertama:")
df_avg.head()


#### 4.3.2 Standardisasi & Elbow Method

In [ ]:
# Standardisasi fitur
features = ['persen_miskin', 'prevalensi_stunting', 'persen_sanitasi']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_avg[features])

# Elbow method & Silhouette score
K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Jumlah Cluster (k)', fontsize=12)
axes[0].set_ylabel('Inertia (SSE)', fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouettes, 'go-', linewidth=2, markersize=8)
axes[1].set_title('Silhouette Score', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Jumlah Cluster (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Tandai k optimal
k_optimal = list(K_range)[np.argmax(silhouettes)]
axes[1].axvline(x=k_optimal, color='red', linestyle='--',
                label=f'Optimal k = {k_optimal}')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nSilhouette Scores:")
for k, s in zip(K_range, silhouettes):
    marker = " <-- OPTIMAL" if k == k_optimal else ""
    print(f"  k={k}: {s:.4f}{marker}")


#### 4.3.3 Fitting K-Means Final

In [ ]:
# Fit K-Means dengan k optimal (gunakan 3 untuk interpretasi bisnis jika optimal bukan 3)
k_final = 3  # 3 cluster: Risiko Tinggi, Menengah, Rendah
print(f"Menggunakan k = {k_final} cluster untuk interpretasi bisnis\n")

km_final = KMeans(n_clusters=k_final, random_state=42, n_init=10, max_iter=300)
df_avg['cluster'] = km_final.fit_predict(X_scaled)

# Hitung silhouette score
sil_final = silhouette_score(X_scaled, km_final.labels_)
print(f"Silhouette Score (k={k_final}): {sil_final:.4f}")

# Statistik per cluster
print("\n" + "="*60)
print("STATISTIK PER CLUSTER")
print("="*60)
cluster_stats = df_avg.groupby('cluster')[features].mean().round(2)
print(cluster_stats)


#### 4.3.4 Auto-Label Cluster

In [ ]:
# Auto-label cluster berdasarkan rata-rata stunting
cluster_means = df_avg.groupby('cluster')['prevalensi_stunting'].mean()
sorted_clusters = cluster_means.sort_values()

label_map = {}
labels_list = ['Risiko Rendah', 'Risiko Menengah', 'Risiko Tinggi']
for i, cluster_id in enumerate(sorted_clusters.index):
    label_map[cluster_id] = labels_list[i]

df_avg['cluster_label'] = df_avg['cluster'].map(label_map)

print("="*60)
print("LABEL CLUSTER")
print("="*60)
for cluster_id, label in sorted(label_map.items()):
    n_prov = len(df_avg[df_avg['cluster'] == cluster_id])
    mean_stunting = cluster_means[cluster_id]
    print(f"  Cluster {cluster_id} -> {label} ({n_prov} provinsi, "
          f"rata-rata stunting: {mean_stunting:.1f}%)")

print("\n" + "="*60)
print("DAFTAR PROVINSI PER CLUSTER")
print("="*60)
for label in labels_list:
    prov_list = df_avg[df_avg['cluster_label'] == label]['provinsi'].tolist()
    print(f"\n{label} ({len(prov_list)} provinsi):")
    for p in sorted(prov_list):
        print(f"  - {p}")


#### 4.3.5 Visualisasi Cluster

In [ ]:
# Visualisasi cluster - 2D scatter plots (multiple projections)
colors_map = {'Risiko Rendah': '#2ecc71', 'Risiko Menengah': '#f39c12',
              'Risiko Tinggi': '#e74c3c'}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Plot 1: Kemiskinan vs Stunting
for label, color in colors_map.items():
    mask = df_avg['cluster_label'] == label
    axes[0].scatter(df_avg.loc[mask, 'persen_miskin'],
                    df_avg.loc[mask, 'prevalensi_stunting'],
                    c=color, label=label, s=80, alpha=0.8, edgecolors='white')
axes[0].set_xlabel('Kemiskinan (%)', fontsize=11)
axes[0].set_ylabel('Stunting (%)', fontsize=11)
axes[0].set_title('Kemiskinan vs Stunting', fontsize=13, fontweight='bold')
axes[0].legend()

# Plot 2: Sanitasi vs Stunting
for label, color in colors_map.items():
    mask = df_avg['cluster_label'] == label
    axes[1].scatter(df_avg.loc[mask, 'persen_sanitasi'],
                    df_avg.loc[mask, 'prevalensi_stunting'],
                    c=color, label=label, s=80, alpha=0.8, edgecolors='white')
axes[1].set_xlabel('Sanitasi (%)', fontsize=11)
axes[1].set_ylabel('Stunting (%)', fontsize=11)
axes[1].set_title('Sanitasi vs Stunting', fontsize=13, fontweight='bold')
axes[1].legend()

# Plot 3: Kemiskinan vs Sanitasi
for label, color in colors_map.items():
    mask = df_avg['cluster_label'] == label
    axes[2].scatter(df_avg.loc[mask, 'persen_miskin'],
                    df_avg.loc[mask, 'persen_sanitasi'],
                    c=color, label=label, s=80, alpha=0.8, edgecolors='white')
axes[2].set_xlabel('Kemiskinan (%)', fontsize=11)
axes[2].set_ylabel('Sanitasi (%)', fontsize=11)
axes[2].set_title('Kemiskinan vs Sanitasi', fontsize=13, fontweight='bold')
axes[2].legend()

plt.suptitle(f'K-Means Clustering (k={k_final}) - Proyeksi 2D\nSilhouette Score: {sil_final:.4f}',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# Simpan model clustering dan data cluster
joblib.dump({'kmeans': km_final, 'scaler': scaler}, get_path('model_kmeans_stunting.pkl', 'models'))
df_avg.to_csv(get_path('dataset_avg_cluster.csv', 'dataset/processed'), index=False)
print("Model K-Means disimpan: model_kmeans_stunting.pkl")
print("Data cluster disimpan: dataset_avg_cluster.csv")


### 4.4 Analisis Tren Temporal (2021–2024)

In [ ]:
# Tren rata-rata nasional per tahun
tren = df_merged.groupby('tahun').agg({
    'prevalensi_stunting': 'mean',
    'persen_miskin': 'mean',
    'persen_sanitasi': 'mean'
}).round(2)

print("="*60)
print("TREN RATA-RATA NASIONAL 2021-2024")
print("="*60)
print(tren)

# Visualisasi tren
fig, ax1 = plt.subplots(figsize=(12, 6))

x = tren.index
ax1.plot(x, tren['prevalensi_stunting'], 'ro-', linewidth=2.5, markersize=10,
         label='Stunting (%)')
ax1.plot(x, tren['persen_miskin'], 'bs-', linewidth=2.5, markersize=10,
         label='Kemiskinan (%)')
ax1.set_xlabel('Tahun', fontsize=12)
ax1.set_ylabel('Persentase (%)', fontsize=12)

# Sumbu kedua untuk sanitasi (skala berbeda)
ax2 = ax1.twinx()
ax2.plot(x, tren['persen_sanitasi'], 'g^-', linewidth=2.5, markersize=10,
         label='Sanitasi (%)')
ax2.set_ylabel('Persentase Sanitasi (%)', fontsize=12, color='green')
ax2.tick_params(axis='y', labelcolor='green')

# Kombinasi legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper center',
           bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=11)

ax1.set_title('Tren Stunting, Kemiskinan, dan Sanitasi (2021-2024)',
              fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## FASE 5: Evaluation

### 5.1 Evaluasi Terhadap Kriteria Keberhasilan


In [ ]:
# Evaluasi kriteria keberhasilan
print("="*60)
print("EVALUASI KRITERIA KEBERHASILAN")
print("="*60)

# Kriteria 1: Korelasi Pearson signifikan
r_km, p_km = stats.pearsonr(df_merged['persen_miskin'], df_merged['prevalensi_stunting'])
r_sn, p_sn = stats.pearsonr(df_merged['persen_sanitasi'], df_merged['prevalensi_stunting'])
k1_pass = p_km < 0.05 and p_sn < 0.05
print(f"\n1. Korelasi Pearson signifikan (p < 0.05):")
print(f"   Kemiskinan-Stunting : r={r_km:.4f}, p={p_km:.6f} -> {'SIGNIFIKAN' if p_km < 0.05 else 'TIDAK'}")
print(f"   Sanitasi-Stunting   : r={r_sn:.4f}, p={p_sn:.6f} -> {'SIGNIFIKAN' if p_sn < 0.05 else 'TIDAK'}")
print(f"   Status: {'✅ TERCAPAI' if k1_pass else '❌ TIDAK TERCAPAI'}")

# Kriteria 2: R² >= 0.30
k2_pass = r2_multi >= 0.30
print(f"\n2. R² Regresi Linear ≥ 0.30:")
print(f"   R² (berganda) = {r2_multi:.4f}")
print(f"   Status: {'✅ TERCAPAI' if k2_pass else '❌ TIDAK TERCAPAI'}")

# Kriteria 3: Silhouette Score >= 0.40
k3_pass = sil_final >= 0.40
print(f"\n3. Silhouette Score Clustering ≥ 0.40:")
print(f"   Silhouette Score = {sil_final:.4f}")
print(f"   Status: {'✅ TERCAPAI' if k3_pass else '❌ TIDAK TERCAPAI'}")

# Kriteria 4: Tren temporal konsisten
print(f"\n4. Identifikasi tren temporal yang konsisten:")
print(f"   Stunting 2021 -> 2024: {tren.loc[2021, 'prevalensi_stunting']:.1f}% -> {tren.loc[2024, 'prevalensi_stunting']:.1f}%")
print(f"   Kemiskinan 2021 -> 2024: {tren.loc[2021, 'persen_miskin']:.1f}% -> {tren.loc[2024, 'persen_miskin']:.1f}%")
print(f"   Sanitasi 2021 -> 2024: {tren.loc[2021, 'persen_sanitasi']:.1f}% -> {tren.loc[2024, 'persen_sanitasi']:.1f}%")
print(f"   Status: ✅ TERCAPAI (tren teridentifikasi)")

total_pass = sum([k1_pass, k2_pass, k3_pass, True])
print(f"\n{'='*60}")
print(f"TOTAL KRITERIA TERCAPAI: {total_pass}/4")
print(f"{'='*60}")


### 5.2 Validasi Temporal (Train 2021–2022, Test 2023–2024)

In [ ]:
# Validasi temporal: train 2021-2022, test 2023-2024
train_mask = df_merged['tahun'].isin([2021, 2022])
test_mask = df_merged['tahun'].isin([2023, 2024])

X_train = df_merged.loc[train_mask, ['persen_miskin', 'persen_sanitasi']].values
y_train = df_merged.loc[train_mask, 'prevalensi_stunting'].values
X_test = df_merged.loc[test_mask, ['persen_miskin', 'persen_sanitasi']].values
y_test = df_merged.loc[test_mask, 'prevalensi_stunting'].values

print("="*60)
print("VALIDASI TEMPORAL")
print("="*60)
print(f"Data latih (2021-2022): {len(X_train)} baris")
print(f"Data uji   (2023-2024): {len(X_test)} baris")

reg_temporal = LinearRegression()
reg_temporal.fit(X_train, y_train)

y_pred_train = reg_temporal.predict(X_train)
y_pred_test = reg_temporal.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"\nHasil Training (2021-2022):")
print(f"  R²   = {r2_train:.4f}")
print(f"  RMSE = {rmse_train:.4f}")

print(f"\nHasil Testing (2023-2024):")
print(f"  R²   = {r2_test:.4f}")
print(f"  RMSE = {rmse_test:.4f}")

gap = abs(r2_train - r2_test)
print(f"\nGap R² (train - test) = {gap:.4f}")
if gap < 0.15:
    print("-> Model cukup stabil (tidak overfitting signifikan)")
else:
    print("-> Perlu perhatian: kemungkinan overfitting atau perubahan pola data")

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(y_train, y_pred_train, alpha=0.5, color='#3498db', edgecolors='white', s=60)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', linewidth=2)
axes[0].set_title(f'Training (2021-2022)\nR² = {r2_train:.4f}', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Aktual (%)')
axes[0].set_ylabel('Prediksi (%)')
axes[0].set_aspect('equal', adjustable='box')

axes[1].scatter(y_test, y_pred_test, alpha=0.5, color='#e67e22', edgecolors='white', s=60)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[1].set_title(f'Testing (2023-2024)\nR² = {r2_test:.4f}', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Aktual (%)')
axes[1].set_ylabel('Prediksi (%)')
axes[1].set_aspect('equal', adjustable='box')

plt.suptitle('Validasi Temporal - Regresi Berganda', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 5.3 Interpretasi Bisnis & Jawaban Pertanyaan Penelitian

In [ ]:
print("="*60)
print("JAWABAN PERTANYAAN PENELITIAN")
print("="*60)

print("\n" + "-"*60)
print("Q1: Apakah kemiskinan dan sanitasi berkorelasi dengan stunting?")
print("-"*60)
print(f"YA. Korelasi Pearson antara kemiskinan dan stunting: r = {r_km:.4f} (p = {p_km:.6f})")
print(f"Korelasi Pearson antara sanitasi dan stunting: r = {r_sn:.4f} (p = {p_sn:.6f})")
if r_km > 0:
    print("Kemiskinan berkorelasi POSITIF dengan stunting (semakin miskin, semakin tinggi stunting)")
if r_sn < 0:
    print("Sanitasi berkorelasi NEGATIF dengan stunting (semakin baik sanitasi, semakin rendah stunting)")

print("\n" + "-"*60)
print("Q2: Seberapa akurat kemiskinan dan sanitasi memprediksi stunting?")
print("-"*60)
print(f"Model regresi berganda menjelaskan {r2_multi*100:.1f}% variasi stunting (R² = {r2_multi:.4f})")
print(f"Model sederhana kemiskinan: R² = {r2_kem:.4f} ({r2_kem*100:.1f}%)")
print(f"Model sederhana sanitasi: R² = {r2_san:.4f} ({r2_san*100:.1f}%)")
print(f"Gabungan kedua variabel meningkatkan akurasi prediksi.")

print("\n" + "-"*60)
print("Q3: Provinsi mana yang memiliki profil serupa?")
print("-"*60)
for label in ['Risiko Tinggi', 'Risiko Menengah', 'Risiko Rendah']:
    subset = df_avg[df_avg['cluster_label'] == label]
    prov_list = ', '.join(sorted(subset['provinsi'].tolist()))
    print(f"\n{label} ({len(subset)} provinsi):")
    print(f"  Rata-rata: Stunting={subset['prevalensi_stunting'].mean():.1f}%, "
          f"Kemiskinan={subset['persen_miskin'].mean():.1f}%, "
          f"Sanitasi={subset['persen_sanitasi'].mean():.1f}%")
    print(f"  Provinsi: {prov_list}")

print("\n" + "-"*60)
print("Q4: Bagaimana tren 2021-2024?")
print("-"*60)
for col, name in [('prevalensi_stunting', 'Stunting'),
                  ('persen_miskin', 'Kemiskinan'),
                  ('persen_sanitasi', 'Sanitasi')]:
    val_2021 = tren.loc[2021, col]
    val_2024 = tren.loc[2024, col]
    change = val_2024 - val_2021
    direction = "TURUN" if change < 0 else "NAIK"
    print(f"  {name}: {val_2021:.1f}% -> {val_2024:.1f}% ({direction} {abs(change):.1f}%)")


---
## FASE 6: Deployment

### 6.1 Artefak yang Dihasilkan


In [ ]:
# Ringkasan artefak yang dihasilkan
print("="*60)
print("ARTEFAK DEPLOYMENT")
print("="*60)

artifacts = {
    'Dataset Final': get_path('dataset_final_kemiskinan_stunting.csv', 'dataset/processed'),
    'Dataset Cluster': get_path('dataset_avg_cluster.csv', 'dataset/processed'),
    'Model Regresi': get_path('model_regresi_stunting.pkl', 'models'),
    'Model K-Means': get_path('model_kmeans_stunting.pkl', 'models')
}

for name, path in artifacts.items():
    print(f"  {name}: {path}")

print("\n" + "="*60)
print("CARA MEMUAT MODEL")
print("="*60)

print("\n# Memuat model regresi:")
print("reg_model = joblib.load('model_regresi_stunting.pkl')")
print("prediksi = reg_model.predict([[persentase_miskin, persentase_sanitasi]])")

print("\n# Memuat model K-Means:")
print("km_artifacts = joblib.load('model_kmeans_stunting.pkl')")
print("km_model = km_artifacts['kmeans']")
print("scaler = km_artifacts['scaler']")
print("cluster = km_model.predict(scaler.transform([[miskin, stunting, sanitasi]]))")


### 6.2 Catatan Deployment

#### Integrasi dengan Streamlit Dashboard
Model dan dataset yang dihasilkan dapat diintegrasikan ke dalam aplikasi **Streamlit** untuk:
1. **Dashboard Interaktif**: Visualisasi peta provinsi berdasarkan cluster
2. **Prediksi Stunting**: Input kemiskinan dan sanitasi → prediksi prevalensi stunting
3. **Monitoring Tren**: Tracking perubahan indikator per provinsi per tahun
4. **Profiling Provinsi**: Detail analisis per provinsi berdasarkan 3 variabel

#### Rekomendasi Kebijakan
Berdasarkan hasil analisis:
1. **Provinsi Risiko Tinggi** perlu intervensi prioritas: program pengentasan kemiskinan + perbaikan sanitasi secara simultan
2. **Provinsi Risiko Menengah** perlu penguatan program yang sudah ada
3. **Provinsi Risiko Rendah** dapat menjadi model *best practice* untuk provinsi lain
4. **Perbaikan sanitasi** memiliki potensi dampak signifikan terhadap penurunan stunting

---
**Notebook ini dibuat menggunakan metodologi CRISP-DM.**
**Data: BPS & Kemenkes RI (2021–2024)**
